In [ ]:
import os, gc, time, random, shutil, psutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image, ImageFile

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b5, EfficientNet_B5_Weights

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, confusion_matrix, precision_recall_curve

from google.colab import drive
from IPython.display import display

# Unmount and remount Google Drive to ensure a fresh connection
if Path('/content/drive').exists():
    print('Unmounting Google Drive...')
    drive.flush_and_unmount()
print('Mounting Google Drive...')
drive.mount('/content/drive')

# ============================================================
# 1) Reproducibility
# ============================================================

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
ImageFile.LOAD_TRUNCATED_IMAGES = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
try:
    torch.set_float32_matmul_precision('medium')
except Exception:
    pass

# ============================================================
# 2) Paths and fixed settings
# ============================================================

COPY_DATASET_TO_LOCAL = True
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/<YOUR_DATASET_FOLDER>/Lusitano_Dataset')
LOCAL_DATASET_ROOT = Path('/content/Lusitano_Dataset')

if COPY_DATASET_TO_LOCAL:
    if not DRIVE_DATASET_ROOT.exists():
        raise ValueError(f'Drive dataset not found: {DRIVE_DATASET_ROOT}')

    expected_local_train = LOCAL_DATASET_ROOT / 'nondefects' / 'nondefects'
    expected_local_test = LOCAL_DATASET_ROOT / 'test' / 'test'

    if expected_local_train.exists() and expected_local_test.exists():
        print('Local dataset already exists:', LOCAL_DATASET_ROOT)
    else:
        if LOCAL_DATASET_ROOT.exists():
            print('Removing incomplete local dataset copy...')
            shutil.rmtree(LOCAL_DATASET_ROOT)

        print('Copying dataset from Google Drive to local Colab storage...')
        print('Source:', DRIVE_DATASET_ROOT)
        print('Target:', LOCAL_DATASET_ROOT)
        copy_start = time.time()
        shutil.copytree(DRIVE_DATASET_ROOT, LOCAL_DATASET_ROOT)
        print(f'Dataset copy completed in {(time.time() - copy_start) / 60:.2f} minutes.')

    DATASET_ROOT = LOCAL_DATASET_ROOT
else:
    DATASET_ROOT = DRIVE_DATASET_ROOT

train_good_path = DATASET_ROOT / 'nondefects' / 'nondefects'
test_root_path  = DATASET_ROOT / 'test' / 'test'

if not train_good_path.exists():
    raise ValueError(f'Training path not found: {train_good_path}')
if not test_root_path.exists():
    raise ValueError(f'Test path not found: {test_root_path}')

IMG_SIZE = 640
METHOD_NAME = f'{IMG_SIZE}px Self-Filtered Greedy PatchCore'
BATCH_SIZE = 16
NUM_WORKERS = 0
PATCHES_PER_IMAGE = 200
PRE_POOL = 400_000
MAX_MEM_PATCHES = 20_000
NN_CHUNK = 40_000
CORESET_CHUNK = 40_000
THRESH_SAMPLE_IMAGES = 2000
THRESHOLD_MODE = 'mean_plus_3std'
FILTER_PERCENTS = [0.0, 3.0, 5.0]

LAYER_COMBOS = {
    'Proposed_EB5_5_7': [5, 7],
    'Baseline_EB5_3_5_7': [3, 5, 7],
}

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is not active. Enable GPU: Runtime > Change runtime type > GPU, then rerun.')

DEVICE = 'cuda'
print('Using device:', DEVICE)
print('GPU:', torch.cuda.get_device_name(0))

SAVE_DIR = Path('/content/drive/MyDrive/<YOUR_OUTPUT_FOLDER>/patchcore_512_layer_filter_comparison')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
ALL_RESULTS_CSV = SAVE_DIR / 'patchcore_512_layer_filter_comparison_all_results_seed42.csv'
BEST_RESULTS_CSV = SAVE_DIR / 'patchcore_512_layer_filter_comparison_best_results_seed42.csv'
FILTER_RANKING_CSV = SAVE_DIR / 'patchcore_512_layer_filter_comparison_train_filter_ranking_seed42.csv'
BEST_IMAGE_SCORES_CSV = SAVE_DIR / 'patchcore_512_layer_filter_comparison_best_image_scores_seed42.csv'

print('Train folder:', train_good_path)
print('Test folder :', test_root_path)
print('Save dir    :', SAVE_DIR)

# ============================================================
# 3) Helpers
# ============================================================

def bytes_to_mb(x):
    return x / (1024 ** 2)

def tensor_size_mb(tensor):
    return bytes_to_mb(tensor.numel() * tensor.element_size())

def model_size_mb(model):
    total_bytes = 0
    for p in model.parameters():
        total_bytes += p.numel() * p.element_size()
    for b in model.buffers():
        total_bytes += b.numel() * b.element_size()
    return bytes_to_mb(total_bytes)

def current_ram_mb():
    process = psutil.Process(os.getpid())
    return bytes_to_mb(process.memory_info().rss)

def reset_peak_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

def get_peak_memory_mb():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        return bytes_to_mb(torch.cuda.max_memory_allocated())
    return current_ram_mb()

def edge_efficiency_score(auc, ap, f1, time_per_image, total_mb, peak_gpu_mb):
    numerator = 0.25 * auc + 0.35 * ap + 0.40 * f1
    denominator = 0.20 * time_per_image + 0.40 * total_mb + 0.40 * peak_gpu_mb
    return 0.0 if denominator <= 0 else numerator / denominator

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

def list_images(folder: Path):
    valid_paths = []
    for p in sorted(folder.glob('*')):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            try:
                Image.open(p).verify()
                valid_paths.append(p)
            except Exception as e:
                print('Corrupted image skipped:', p, e)
    return valid_paths

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class ImagePathDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = list(paths)
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        p = self.paths[idx]
        img = Image.open(p).convert('RGB')
        return self.transform(img), str(p)

class TestImageDataset(Dataset):
    def __init__(self, items, transform):
        self.items = list(items)
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        p, label = self.items[idx]
        img = Image.open(p).convert('RGB')
        return self.transform(img), int(label), str(p)

def make_image_loader(paths, batch_size=BATCH_SIZE, shuffle=False):
    return DataLoader(
        ImagePathDataset(paths, transform),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == 'cuda'),
        drop_last=False,
    )

def make_test_loader(items, batch_size=BATCH_SIZE):
    return DataLoader(
        TestImageDataset(items, transform),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == 'cuda'),
        drop_last=False,
    )

# ============================================================
# 4) EfficientNet-B5 feature extractor
# ============================================================

class EfficientNetFeatureExtractor(torch.nn.Module):
    def __init__(self, layer_indices):
        super().__init__()
        self.layer_indices = list(layer_indices)
        self.model = efficientnet_b5(weights=EfficientNet_B5_Weights.DEFAULT)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad = False
        self.features = []
        self.hooks = []
        def make_hook():
            def hook(_, __, output):
                self.features.append(output)
            return hook
        for idx in self.layer_indices:
            self.hooks.append(self.model.features[idx].register_forward_hook(make_hook()))

    def forward(self, x):
        self.features = []
        with torch.no_grad():
            _ = self.model(x)
        if len(self.features) != len(self.layer_indices):
            raise RuntimeError(f'Expected {len(self.layer_indices)} feature maps, got {len(self.features)}')
        fmap_size = min(f.shape[-2] for f in self.features)
        resize = torch.nn.AdaptiveAvgPool2d(fmap_size)
        resized = [resize(f) for f in self.features]
        patch_features = torch.cat(resized, dim=1)
        B, C, H, W = patch_features.shape
        return patch_features.reshape(B, C, H * W).permute(0, 2, 1)

# ============================================================
# 5) Load dataset
# ============================================================

train_paths = list_images(train_good_path)
if len(train_paths) == 0:
    raise ValueError('No training images found.')
print('\nTraining normal images:', len(train_paths))

def get_label(folder_name):
    name = folder_name.lower().replace('_', '-').strip()
    if name == 'non-defects':
        return 0
    if name == 'defects':
        return 1
    return None

test_items = []
for folder in sorted(test_root_path.iterdir()):
    if not folder.is_dir():
        continue
    label = get_label(folder.name)
    if label is None:
        print('Skipping unknown folder:', folder.name)
        continue
    for p in list_images(folder):
        test_items.append((p, label))

if len(test_items) == 0:
    raise ValueError('No test images found.')

print('Total test images:', len(test_items))
print('Normal test images:', sum(1 for _, y in test_items if y == 0))
print('Defect test images:', sum(1 for _, y in test_items if y == 1))
test_loader = make_test_loader(test_items)

# ============================================================
# 6) Candidate pool and Greedy Coreset
# ============================================================

@torch.no_grad()
def extract_candidate_pool(backbone, paths, desc='Extracting candidate patch pool'):
    set_seed(SEED)
    loader = make_image_loader(paths, batch_size=BATCH_SIZE, shuffle=False)
    all_features = []
    print(f'\n{desc}')
    for xb, _ in tqdm(loader, desc=desc):
        xb = xb.to(DEVICE, non_blocking=True)
        feats = backbone(xb)
        B, N, C = feats.shape
        for i in range(B):
            n = min(PATCHES_PER_IMAGE, N)
            idx = torch.randperm(N, device=DEVICE)[:n]
            all_features.append(feats[i, idx].detach().float().cpu())
        del xb, feats
    candidate_pool = torch.cat(all_features, dim=0)
    del all_features
    print('Candidate pool before PRE_POOL cap:', candidate_pool.shape)
    if candidate_pool.shape[0] > PRE_POOL:
        set_seed(SEED)
        idx = torch.randperm(candidate_pool.shape[0])[:PRE_POOL]
        candidate_pool = candidate_pool[idx]
    print('Final candidate pool:', candidate_pool.shape)
    return candidate_pool

@torch.no_grad()
def greedy_coreset_gpu(features_cpu, max_samples, chunk=40_000, use_fp16=True, device='cuda', seed=42):
    random.seed(seed)
    N, C = features_cpu.shape
    if N <= max_samples:
        return features_cpu.clone()
    feats = features_cpu.to(device, non_blocking=True).contiguous()
    if use_fp16:
        feats = feats.half()
    selected_idx = torch.empty((max_samples,), dtype=torch.long, device=device)
    first = random.randint(0, N - 1)
    selected_idx[0] = first
    center = feats[first:first + 1]
    min_d = torch.empty((N,), device=device, dtype=torch.float32)
    for start in range(0, N, chunk):
        x = feats[start:start + chunk]
        min_d[start:start + chunk] = (x - center).float().pow(2).sum(dim=1)
    for i in tqdm(range(1, max_samples), desc='Greedy coreset'):
        farthest = torch.argmax(min_d).item()
        selected_idx[i] = farthest
        center = feats[farthest:farthest + 1]
        for start in range(0, N, chunk):
            x = feats[start:start + chunk]
            d = (x - center).float().pow(2).sum(dim=1)
            min_d[start:start + chunk] = torch.minimum(min_d[start:start + chunk], d)
    selected = feats[selected_idx].float().cpu()
    del feats, selected_idx, min_d
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return selected

# ============================================================
# 7) Scoring and evaluation
# ============================================================

@torch.no_grad()
def image_anomaly_score(patch_feats_gpu, memory_bank_gpu, chunk_size=40_000):
    x = patch_feats_gpu.float()
    P = x.shape[0]
    min_dist = torch.full((P,), float('inf'), device=DEVICE, dtype=torch.float32)
    x2 = x.pow(2).sum(dim=1, keepdim=True)
    for start in range(0, memory_bank_gpu.shape[0], chunk_size):
        mb = memory_bank_gpu[start:start + chunk_size].float()
        mb2 = mb.pow(2).sum(dim=1).unsqueeze(0)
        d2 = x2 + mb2 - 2.0 * (x @ mb.t())
        d2 = torch.clamp(d2, min=0.0)
        min_dist = torch.minimum(min_dist, d2.min(dim=1).values)
    return min_dist.sqrt().max().item()

@torch.no_grad()
def score_paths(backbone, paths, memory_bank_gpu, desc='Scoring images'):
    loader = make_image_loader(paths, batch_size=BATCH_SIZE, shuffle=False)
    scores, scored_paths = [], []
    for xb, paths_batch in tqdm(loader, desc=desc):
        xb = xb.to(DEVICE, non_blocking=True)
        feats = backbone(xb)
        for i in range(feats.shape[0]):
            scores.append(float(image_anomaly_score(feats[i], memory_bank_gpu, chunk_size=NN_CHUNK)))
            scored_paths.append(str(paths_batch[i]))
        del xb, feats
    return np.array(scores, dtype=np.float32), scored_paths

@torch.no_grad()
def compute_threshold(backbone, clean_train_paths, memory_bank_gpu):
    rng = np.random.default_rng(SEED + 100)
    num_for_thresh = min(THRESH_SAMPLE_IMAGES, len(clean_train_paths))
    thresh_indices = rng.permutation(len(clean_train_paths))[:num_for_thresh]
    thresh_subset = [clean_train_paths[i] for i in thresh_indices]

    threshold_scores, _ = score_paths(
        backbone,
        thresh_subset,
        memory_bank_gpu,
        desc='Threshold scoring'
    )

    if THRESHOLD_MODE == 'mean_plus_3std':
        threshold = threshold_scores.mean() + 3.0 * threshold_scores.std(ddof=1)
    else:
        raise ValueError(f'Unknown THRESHOLD_MODE: {THRESHOLD_MODE}')

    return float(threshold), threshold_scores

@torch.no_grad()
def evaluate_test_set(backbone, memory_bank_gpu, threshold):
    reset_peak_memory()

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    end_to_end_start = time.perf_counter()
    compute_total_time = 0.0

    y_true, y_score, image_paths = [], [], []

    print('\nEvaluating test set...')

    for xb, yb, paths_batch in tqdm(test_loader, desc='Testing'):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        compute_start = time.perf_counter()

        xb = xb.to(DEVICE, non_blocking=True)
        feats = backbone(xb)

        batch_scores = []
        for i in range(feats.shape[0]):
            batch_scores.append(
                float(image_anomaly_score(feats[i], memory_bank_gpu, chunk_size=NN_CHUNK))
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        compute_total_time += time.perf_counter() - compute_start

        y_score.extend(batch_scores)
        y_true.extend([int(v) for v in yb])
        image_paths.extend([str(p) for p in paths_batch])

        del xb, feats, batch_scores

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    end_to_end_total_time = time.perf_counter() - end_to_end_start
    compute_inference_time_per_image = compute_total_time / max(len(y_true), 1)
    end_to_end_local_runtime_per_image = end_to_end_total_time / max(len(y_true), 1)
    peak_memory_usage_mb = get_peak_memory_mb()

    y_true = np.array(y_true, dtype=np.int32)
    y_score = np.array(y_score, dtype=np.float32)

    y_pred = (y_score > threshold).astype(np.int32)

    auc_roc = roc_auc_score(y_true, y_score)
    map_ap = average_precision_score(y_true, y_score)
    f1 = f1_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    precision, recall, pr_thresholds = precision_recall_curve(y_true, y_score)
    f1s = 2.0 * precision * recall / (precision + recall + 1e-12)
    best_idx = int(np.argmax(f1s))
    best_f1 = float(f1s[best_idx])
    best_precision = float(precision[best_idx])
    best_recall = float(recall[best_idx])
    best_threshold = float(y_score.min()) if best_idx == 0 else float(pr_thresholds[best_idx - 1])

    df_scores = pd.DataFrame({
        'image_path': image_paths,
        'gt_label': y_true,
        'anomaly_score': y_score,
        'pred_label': y_pred,
    })

    return {
        'AUC_ROC': float(auc_roc),
        'mAP_AP': float(map_ap),
        'F1_Score': float(f1),
        'Best_F1_Diagnostic': best_f1,
        'Best_F1_Threshold': best_threshold,
        'Best_F1_Precision': best_precision,
        'Best_F1_Recall': best_recall,
        'Threshold': float(threshold),
        'TN': int(tn),
        'FP': int(fp),
        'FN': int(fn),
        'TP': int(tp),
        'Inference_Time_Per_Image_sec': float(compute_inference_time_per_image),
        'Compute_Inference_Time_Per_Image_sec': float(compute_inference_time_per_image),
        'End_To_End_Local_Runtime_Per_Image_sec': float(end_to_end_local_runtime_per_image),
        'Compute_Total_Time_sec': float(compute_total_time),
        'End_To_End_Local_Total_Time_sec': float(end_to_end_total_time),
        'Peak_Memory_Usage_MB': float(peak_memory_usage_mb),
    }, df_scores

# ============================================================
# 8) Self-filter utilities
# ============================================================

def make_filter_ranking(train_paths, train_scores, layer_name, layer_indices):
    df = pd.DataFrame({
        'image_path': [str(p) for p in train_paths],
        'initial_greedy_train_score': train_scores,
        'Layer_Combo': layer_name,
        'Layer_Indices': str(layer_indices),
        'IMG_SIZE': IMG_SIZE,
    })
    df = df.sort_values('initial_greedy_train_score', ascending=False).reset_index(drop=True)
    df['Filter_Rank'] = np.arange(1, len(df) + 1)
    return df

def get_clean_train_paths(filter_df, filter_percent):
    df = filter_df.copy()
    n_total = len(df)
    df['filtered_out'] = False
    if filter_percent <= 0:
        return [Path(p) for p in df['image_path'].tolist()], [], df
    n_remove = int(round(n_total * filter_percent / 100.0))
    n_remove = max(0, min(n_remove, n_total - 1))
    if n_remove > 0:
        df.loc[:n_remove - 1, 'filtered_out'] = True
    removed_paths = [Path(p) for p in df.loc[df['filtered_out'], 'image_path'].tolist()]
    clean_paths = [Path(p) for p in df.loc[~df['filtered_out'], 'image_path'].tolist()]
    return clean_paths, removed_paths, df

# ============================================================
# 9) Full experiment for one layer setting
# ============================================================

def run_layer_experiment(layer_name, layer_indices):
    print('\n' + '=' * 100)
    print(f'Layer experiment: {layer_name} | layers={layer_indices} | IMG_SIZE={IMG_SIZE}')
    print('=' * 100)
    set_seed(SEED)
    reset_peak_memory()
    backbone = EfficientNetFeatureExtractor(layer_indices=layer_indices).to(DEVICE).eval()
    with torch.no_grad():
        dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
        dummy_feats = backbone(dummy)
        D = int(dummy_feats.shape[-1])
        Patch_Grid_N = int(dummy_feats.shape[1])
        del dummy, dummy_feats
    backbone_size_mb = model_size_mb(backbone)
    print('D:', D)
    print('Patch_Grid_N:', Patch_Grid_N)
    print(f'Backbone MB: {backbone_size_mb:.3f}')

    t0 = time.time()
    candidate_pool = extract_candidate_pool(backbone, train_paths, desc=f'Initial candidate pool | {layer_name}')
    initial_pool_shape = tuple(candidate_pool.shape)
    print('\nBuilding initial Greedy Coreset memory bank...')
    initial_memory_bank_cpu = greedy_coreset_gpu(candidate_pool, MAX_MEM_PATCHES, CORESET_CHUNK, True, DEVICE, SEED)
    del candidate_pool
    gc.collect(); torch.cuda.empty_cache()
    initial_memory_bank_gpu = initial_memory_bank_cpu.to(DEVICE, non_blocking=True)
    initial_build_time_sec = time.time() - t0
    print('Initial memory shape:', tuple(initial_memory_bank_cpu.shape))
    print(f'Initial memory MB: {tensor_size_mb(initial_memory_bank_cpu):.3f}')

    t_score = time.time()
    train_scores, _ = score_paths(backbone, train_paths, initial_memory_bank_gpu, desc=f'Train-normal scoring | {layer_name}')
    train_scoring_time_sec = time.time() - t_score
    filter_df_base = make_filter_ranking(train_paths, train_scores, layer_name, layer_indices)
    del initial_memory_bank_gpu
    gc.collect(); torch.cuda.empty_cache()

    all_rows, all_filter_dfs, score_df_map = [], [], {}
    for filter_percent in FILTER_PERCENTS:
        print('\n' + '-' * 100)
        print(f'Final evaluation | {layer_name} | filter={filter_percent}% | IMG_SIZE={IMG_SIZE}')
        print('-' * 100)
        clean_train_paths, removed_paths, filter_df = get_clean_train_paths(filter_df_base, filter_percent)
        filter_df['Filter_Percent'] = float(filter_percent)
        all_filter_dfs.append(filter_df)
        print('Original train normals:', len(train_paths))
        print('Removed normals       :', len(removed_paths))
        print('Clean train normals   :', len(clean_train_paths))

        final_build_start = time.time()
        if filter_percent == 0:
            final_memory_bank_cpu = initial_memory_bank_cpu.clone()
            final_candidate_pool_shape = initial_pool_shape
        else:
            clean_candidate_pool = extract_candidate_pool(backbone, clean_train_paths, desc=f'Clean candidate pool | {layer_name} | filter={filter_percent}%')
            final_candidate_pool_shape = tuple(clean_candidate_pool.shape)
            print('\nBuilding final Greedy Coreset memory bank from cleaned normals...')
            final_memory_bank_cpu = greedy_coreset_gpu(clean_candidate_pool, MAX_MEM_PATCHES, CORESET_CHUNK, True, DEVICE, SEED)
            del clean_candidate_pool
            gc.collect(); torch.cuda.empty_cache()
        final_build_time_sec = time.time() - final_build_start
        memory_bank_size_mb = tensor_size_mb(final_memory_bank_cpu)
        estimated_total_footprint_mb = memory_bank_size_mb + backbone_size_mb
        final_memory_bank_gpu = final_memory_bank_cpu.to(DEVICE, non_blocking=True)
        print('Final memory shape:', tuple(final_memory_bank_cpu.shape))
        print(f'Memory bank MB: {memory_bank_size_mb:.3f}')
        print(f'Total footprint MB: {estimated_total_footprint_mb:.3f}')

        threshold, _ = compute_threshold(backbone, clean_train_paths, final_memory_bank_gpu)
        print(f'Internal threshold for F1: {threshold:.6f}')
        eval_result, df_image_scores = evaluate_test_set(backbone, final_memory_bank_gpu, threshold)
        edge_eff = edge_efficiency_score(eval_result['AUC_ROC'], eval_result['mAP_AP'], eval_result['F1_Score'], eval_result['Inference_Time_Per_Image_sec'], estimated_total_footprint_mb, eval_result['Peak_Memory_Usage_MB'])

        row = {
            'Dataset': 'Lusitano', 'Method': METHOD_NAME, 'Backbone': 'EfficientNet-B5',
            'Layer_Combo': layer_name, 'Layer_Indices': str(layer_indices), 'Seed': SEED,
            'IMG_SIZE': IMG_SIZE, 'D': D, 'Patch_Grid_N': Patch_Grid_N,
            'PATCHES_PER_IMAGE': PATCHES_PER_IMAGE, 'PRE_POOL': PRE_POOL, 'MAX_MEM_PATCHES': MAX_MEM_PATCHES,
            'NN_CHUNK': NN_CHUNK,
            'CORESET_CHUNK': CORESET_CHUNK,
            'THRESHOLD_MODE': THRESHOLD_MODE,
            'THRESH_SAMPLE_IMAGES': THRESH_SAMPLE_IMAGES,
            'Initial_Candidate_Pool_Shape': str(initial_pool_shape), 'Final_Candidate_Pool_Shape': str(final_candidate_pool_shape),
            'Filter_Percent': float(filter_percent), 'Train_Normal_Original': len(train_paths),
            'Train_Normal_Removed': len(removed_paths), 'Train_Normal_Clean': len(clean_train_paths),
            'AUC_ROC': eval_result['AUC_ROC'], 'mAP_AP': eval_result['mAP_AP'], 'F1_Score': eval_result['F1_Score'],
            'Best_F1_Diagnostic': eval_result['Best_F1_Diagnostic'],
            'Best_F1_Threshold': eval_result['Best_F1_Threshold'],
            'Best_F1_Precision': eval_result['Best_F1_Precision'],
            'Best_F1_Recall': eval_result['Best_F1_Recall'],
            'Threshold': eval_result['Threshold'], 'TN': eval_result['TN'], 'FP': eval_result['FP'],
            'FN': eval_result['FN'], 'TP': eval_result['TP'],
            'Inference_Time_Per_Image_sec': eval_result['Inference_Time_Per_Image_sec'],
            'Compute_Inference_Time_Per_Image_sec': eval_result['Compute_Inference_Time_Per_Image_sec'],
            'End_To_End_Local_Runtime_Per_Image_sec': eval_result['End_To_End_Local_Runtime_Per_Image_sec'],
            'Compute_Total_Time_sec': eval_result['Compute_Total_Time_sec'],
            'End_To_End_Local_Total_Time_sec': eval_result['End_To_End_Local_Total_Time_sec'],
            'Memory_Bank_Size_MB': memory_bank_size_mb, 'Backbone_Model_Size_MB': backbone_size_mb,
            'Estimated_Total_Footprint_MB': estimated_total_footprint_mb,
            'Peak_Memory_Usage_MB': eval_result['Peak_Memory_Usage_MB'],
            'Edge_Efficiency': edge_eff,
            'Initial_Coreset_Build_Time_sec': initial_build_time_sec,
            'Train_Normal_Filter_Scoring_Time_sec': train_scoring_time_sec,
            'Final_Coreset_Build_Time_sec': final_build_time_sec,
            'Test_Total_Time_sec': eval_result['End_To_End_Local_Total_Time_sec'],
        }
        all_rows.append(row)

        score_key = f'{layer_name}_img{IMG_SIZE}_filter_{filter_percent}'
        df_image_scores['Layer_Combo'] = layer_name
        df_image_scores['Layer_Indices'] = str(layer_indices)
        df_image_scores['IMG_SIZE'] = int(IMG_SIZE)
        df_image_scores['Filter_Percent'] = float(filter_percent)
        score_df_map[score_key] = df_image_scores

        print('\nResult')
        print(f"AUC: {eval_result['AUC_ROC']:.6f}")
        print(f"AP : {eval_result['mAP_AP']:.6f}")
        print(f"F1 : {eval_result['F1_Score']:.6f}")
        print(f"Best-F1 diagnostic: {eval_result['Best_F1_Diagnostic']:.6f}")
        print(f"TN FP FN TP: {eval_result['TN']} {eval_result['FP']} {eval_result['FN']} {eval_result['TP']}")
        print(f"Compute time/image: {eval_result['Compute_Inference_Time_Per_Image_sec']:.6f} sec")
        print(f"End-to-end time/image: {eval_result['End_To_End_Local_Runtime_Per_Image_sec']:.6f} sec")
        print(f'Memory bank MB: {memory_bank_size_mb:.3f}')
        print(f'Total footprint MB: {estimated_total_footprint_mb:.3f}')
        print(f"Peak memory MB: {eval_result['Peak_Memory_Usage_MB']:.3f}")
        print(f'Edge efficiency: {edge_eff:.10f}')

        del final_memory_bank_gpu, final_memory_bank_cpu
        gc.collect(); torch.cuda.empty_cache()

    del initial_memory_bank_cpu, backbone
    gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(all_rows), pd.concat(all_filter_dfs, ignore_index=True), score_df_map

# ============================================================
# 10) Run all experiments
# ============================================================

all_results, all_filter_rankings, all_image_score_maps = [], [], {}
global_start = time.time()

for layer_name, layer_indices in LAYER_COMBOS.items():
    df_layer, df_filter_layer, image_score_map = run_layer_experiment(layer_name, layer_indices)
    all_results.append(df_layer)
    all_filter_rankings.append(df_filter_layer)
    all_image_score_maps.update(image_score_map)
    pd.concat(all_results, ignore_index=True).to_csv(ALL_RESULTS_CSV, index=False)
    pd.concat(all_filter_rankings, ignore_index=True).to_csv(FILTER_RANKING_CSV, index=False)

global_total_time_sec = time.time() - global_start

df_all = pd.concat(all_results, ignore_index=True)
max_eff = df_all['Edge_Efficiency'].max()
df_all['Edge_Efficiency_%'] = (df_all['Edge_Efficiency'] / max_eff) * 100.0 if max_eff > 0 else 0.0
df_all.to_csv(ALL_RESULTS_CSV, index=False)

df_filter_all = pd.concat(all_filter_rankings, ignore_index=True)
df_filter_all.to_csv(FILTER_RANKING_CSV, index=False)

best_by_ap = df_all.sort_values('mAP_AP', ascending=False).iloc[0]
best_by_f1 = df_all.sort_values('F1_Score', ascending=False).iloc[0]
best_by_edge = df_all.sort_values('Edge_Efficiency', ascending=False).iloc[0]

df_best = pd.DataFrame([
    {**best_by_ap.to_dict(), 'Selection': 'Best_by_AP'},
    {**best_by_f1.to_dict(), 'Selection': 'Best_by_F1'},
    {**best_by_edge.to_dict(), 'Selection': 'Best_by_Edge_Efficiency'},
])
df_best.to_csv(BEST_RESULTS_CSV, index=False)

best_key = f"{best_by_ap['Layer_Combo']}_img{int(best_by_ap['IMG_SIZE'])}_filter_{best_by_ap['Filter_Percent']}"
if best_key in all_image_score_maps:
    all_image_score_maps[best_key].to_csv(BEST_IMAGE_SCORES_CSV, index=False)
else:
    print('Warning: best image-score key not found:', best_key)

print('\n' + '=' * 100)
print(f'FINAL {IMG_SIZE}PX SELF-FILTERED PATCHCORE LAYER/FILTER COMPARISON SUMMARY')
print('=' * 100)

print('\nAll results sorted by AP')
display(df_all.sort_values('mAP_AP', ascending=False))

print('\nAll results sorted by F1')
display(df_all.sort_values('F1_Score', ascending=False))

print('\nAll results sorted by Edge Efficiency')
display(df_all.sort_values('Edge_Efficiency', ascending=False))

print('\nBest by AP')
display(pd.DataFrame([best_by_ap]))

print('\nBest by F1')
display(pd.DataFrame([best_by_f1]))

print('\nBest by Edge Efficiency')
display(pd.DataFrame([best_by_edge]))

print('\nSaved all results:')
print(ALL_RESULTS_CSV)
print('\nSaved best results:')
print(BEST_RESULTS_CSV)
print('\nSaved train-normal filter ranking:')
print(FILTER_RANKING_CSV)
print('\nSaved best image-level scores:')
print(BEST_IMAGE_SCORES_CSV)
print(f'\nTotal experiment time sec: {global_total_time_sec:.3f}')
print('=' * 100)
